# OncoHack starter: Wisconsin Breast Cancer (Original)

**This is a data-readiness exercise. Building classifiers or predicting diagnosis is out of scope.**

**Shared question (both tracks; see your track folder for the full brief):** A reviewer wants to report one malignant share across all eight collection batches. Should they report the pooled figure, show batch results alongside it, or pause? Show the evidence behind your decision and explain what the data can support.

This notebook only loads the frozen file and derives batch labels. The analysis is yours.

In [ ]:
# Works from a fresh clone (run from the repo root or starter/) and in Google Colab.
# In Colab, the files are downloaded from the public repository. Downloading needs the repo to be public.
import hashlib, pathlib, urllib.request
import pandas as pd

FILES = {
    "breast-cancer-wisconsin.data":  "402c585309c399237740f635ef9919dc512cca12cbeb20de5e563a4593f22b64",
    "breast-cancer-wisconsin.names": "4bcd6e4e0e4f8ae54e313ebbf18013cc83b6827a207123707687f8f19d29dd2c",
}
RAW = "https://raw.githubusercontent.com/disleveK/oncosys-hackathon-challenge/main/data/"

candidates = [pathlib.Path("data"), pathlib.Path("../data")]
DATA_DIR = next((d for d in candidates if (d / "breast-cancer-wisconsin.data").exists()), None)
if DATA_DIR is None:                                   # e.g. Colab: no local copy yet
    DATA_DIR = pathlib.Path("data"); DATA_DIR.mkdir(exist_ok=True)
    for name in FILES:
        urllib.request.urlretrieve(RAW + name, DATA_DIR / name)

for name, expected in FILES.items():
    actual = hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
    assert actual == expected, f"{name} does not match the frozen version: {actual}"
PATH, NAMES = DATA_DIR / "breast-cancer-wisconsin.data", DATA_DIR / "breast-cancer-wisconsin.names"
print("Checksums OK:", DATA_DIR.resolve())

In [ ]:
COLS = ["sample_code", "clump_thickness", "uniformity_cell_size", "uniformity_cell_shape",
        "marginal_adhesion", "single_epithelial_cell_size", "bare_nuclei", "bland_chromatin",
        "normal_nucleoli", "mitoses", "class"]          # class: 2 = benign, 4 = malignant
df = pd.read_csv(PATH, header=None, names=COLS, na_values="?")   # '?' marks a missing value
df["row"] = range(1, len(df) + 1)
df.shape

## Deriving batch labels

The file has **no batch column**. UCI's documentation (`breast-cancer-wisconsin.names`) says the rows arrived in eight groups, in file order, with these sizes. The labels below are therefore *inferred from row position*. Record that assumption in your evidence.

In [ ]:
BATCHES = [("1", "Jan 1989", 367), ("2", "Oct 1989", 70), ("3", "Feb 1990", 31), ("4", "Apr 1990", 17),
           ("5", "Aug 1990", 48), ("6", "Jan 1991", 49), ("7", "Jun 1991", 31), ("8", "Nov 1991", 86)]
assert len(df) == sum(n for _, _, n in BATCHES)
df["batch"] = [b for b, _, n in BATCHES for _ in range(n)]
df.groupby("batch").size()

## Your turn

Some starting points, not a checklist: missing values, repeated sample codes, how batches differ, and how sensitive any headline figure is to the choices you make. Remember that a sample code identifies a *sample record*, not necessarily a patient.